In [36]:
%pip install phonenumbers --quiet

Note: you may need to restart the kernel to use updated packages.


In [37]:
import re
import phonenumbers

Xử lý cho czechia

In [38]:
import pandas as pd

# Đường dẫn tới 2 file
gd_path = r"C:\Users\Nhung\Downloads\We_Love_Pho\sample structure.csv"
checkpoint_path = r"C:\Users\Nhung\Downloads\We_Love_Pho\raw_country_extracted\United_Kingdom.csv"

# 1. Đọc dữ liệu
df_checkpoint = pd.read_csv(checkpoint_path)
df_gd = pd.read_csv(gd_path)

In [39]:
df_checkpoint.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1915 entries, 0 to 1914
Data columns (total 14 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   post_title          1915 non-null   object 
 1   address             1915 non-null   object 
 2   latitude            1915 non-null   float64
 3   longitude           1915 non-null   float64
 4   phone               1796 non-null   object 
 5   website             1540 non-null   object 
 6   facebook            0 non-null      float64
 7   instagram           0 non-null      float64
 8   twitter             0 non-null      float64
 9   post_content        1915 non-null   object 
 10  google_maps_link    1915 non-null   object 
 11  google_profile      1915 non-null   object 
 12  google_review_link  1915 non-null   object 
 13  country             1915 non-null   object 
dtypes: float64(5), object(9)
memory usage: 209.6+ KB


In [40]:
# --- Kiểm tra định dạng số điện thoại trước khi chuẩn hóa ---
phones = df_checkpoint["phone"].astype(str).str.strip()

# Loại bỏ các nan
valid_phones = phones[~phones.str.lower().isin(["nan", "none", ""]) & (phones != "")]

# Kiểm tra định dạng số điện thoại
# Có bắt đầu bằng '00'?
count_00 = valid_phones.str.startswith("00").sum()
# Có bắt đầu bằng '+' ?
count_plus = valid_phones.str.startswith("+").sum()
# Có dấu '-'?
count_dash = valid_phones.str.contains("-", regex=False).sum()
# Có dấu cách ?
count_space = valid_phones.str.contains(" ", regex=False).sum()

# Kiểm tra range số điện thoại
phones_cleaned = valid_phones.str.replace(r"[-\s]", "", regex=True)
lengths = phones_cleaned.str.len()
min_len = lengths.min()
max_len = lengths.max()

# --- In kết quả ---
print(f"Số bắt đầu bằng '00': {count_00}")
print(f"Số bắt đầu bằng '+': {count_plus}")
print(f"Số chứa dấu '-': {count_dash}")
print(f"Số chứa dấu cách: {count_space}")
print(f"Độ dài ngắn nhất (sau khi loại bỏ ký tự và NaN): {min_len}")
print(f"Độ dài dài nhất: {max_len}")


Số bắt đầu bằng '00': 0
Số bắt đầu bằng '+': 0
Số chứa dấu '-': 0
Số chứa dấu cách: 1796
Độ dài ngắn nhất (sau khi loại bỏ ký tự và NaN): 10
Độ dài dài nhất: 11


In [41]:
# Chuẩn hóa số điện thoại theo tiêu chuẩn E.164
from phonenumbers import PhoneNumberFormat
# ---- Country to Region Code Mapping ----
country_region_map = {
    "Sweden": "SE",
    "United Kingdom": "GB",
    "France": "FR",
    "Germany": "DE",
    "Poland": "PL",
    "Czechia": "CZ",
    "Slovakia": "SK",
    "Italy": "IT",
    "Spain": "ES",
    "Portugal": "PT",
    "Belgium": "BE",
    "Netherlands": "NL",
    "Hungary": "HU",
    "Austria": "AT"
}

# Hàm clean - giữ nan và xóa kí tự lạ (gồm dấu cách và - )
def clean_phone_number(raw_phone):
    if pd.isna(raw_phone):
        return raw_phone  
    raw_phone = str(raw_phone)
    cleaned = re.sub(r'[^\d+]', '', raw_phone)  
    return cleaned

# Chuẩn hóa số hợp lệ theo E.164 (thư viện phonenumbers để đưa về dạng sđt quốc tế)
def standardize_phone_number(row):
    raw = clean_phone_number(row["phone"])
    region = country_region_map.get(row["country"], None)
    if pd.isna(raw) or not str(raw).strip():
        return row["phone"]  
    try:
        parsed = phonenumbers.parse(raw, region)
        if phonenumbers.is_valid_number(parsed):
            return phonenumbers.format_number(parsed, PhoneNumberFormat.E164)
        else:
            return row["phone"]
    except:
        return row["phone"]

# Gán nhãn hợp lệ / không hợp lệ / thiếu để tiện lọc thủ công (nếu có)
def label_phone_status(row):
    raw = clean_phone_number(row["phone"])
    region = country_region_map.get(row["country"], None)
    if pd.isna(raw) or not str(raw).strip():
        return pd.NA 
    try:
        parsed = phonenumbers.parse(raw, region)
        if phonenumbers.is_valid_number(parsed):
            return 1  # Valid
        else:
            return 0  # Invalid
    except:
        return 0  # Invalid do lỗi

# Áp dụng hàm
df_checkpoint["phone"] = df_checkpoint.apply(standardize_phone_number, axis=1)
df_checkpoint["phone_status"] = df_checkpoint.apply(label_phone_status, axis=1)
print(df_checkpoint[["phone", "country", "phone_status"]].head(5))

           phone         country phone_status
0  +442074944555  United Kingdom            1
1  +442073513843  United Kingdom            1
2  +442033028828  United Kingdom            1
3  +442088545588  United Kingdom            1
4  +442074371351  United Kingdom            1


In [42]:
df_checkpoint.head(5)

,post_title,address,latitude,longitude,phone,website,facebook,instagram,twitter,post_content,google_maps_link,google_profile,google_review_link,country,phone_status
0,Viet Food,"34-36 Wardour St, London W1D 6QT, UK",51.511621,-0.132167,+442074944555,http://www.vietnamfood.co.uk/,NaN,NaN,NaN,Close this moduleMain CourseFree Range Chicken...,https://maps.google.com/?cid=12549996454222453965,https://maps.google.com/?q=place_id:ChIJe6Owst...,https://search.google.com/local/reviews?placei...,United Kingdom,1
1,Phat Phuc Noodle Bar,"The Courtyard, 151 Sydney St, London SW3 6NT, UK",51.487544,-0.169204,+442073513843,http://www.phatphucnoodlebar.co.uk/,NaN,NaN,NaN,"Phat Phuc Noodle Bar located in The Courtyard,...",https://maps.google.com/?cid=18023201915501093142,https://maps.google.com/?q=place_id:ChIJwbvREG...,https://search.google.com/local/reviews?placei...,United Kingdom,1
2,Union Viet Café,"120 Union St, London SE1 0FR, UK",51.503872,-0.098374,+442033028828,http://www.unionviet.com/,NaN,NaN,NaN,Union Viet family run cafe & pho restaurant. ...,https://maps.google.com/?cid=5282475706193879885,https://maps.google.com/?q=place_id:ChIJuXu1JK...,https://search.google.com/local/reviews?placei...,United Kingdom,1
3,Viet Baguette (Pho & Grill),"17 Anglesea Rd, London SE18 6EG, UK",51.488533,0.067871,+442088545588,NaN,NaN,NaN,NaN,Viet Baguette (Pho & Grill) located in 17 Angl...,https://maps.google.com/?cid=14466978752506452667,https://maps.google.com/?q=place_id:ChIJhcPehu...,https://search.google.com/local/reviews?placei...,United Kingdom,1
4,Banana Tree Soho,"103 Wardour St, London W1F 0UG, UK",51.513099,-0.133865,+442074371351,https://bananatree.co.uk/restaurants/soho/?utm...,NaN,NaN,NaN,Banana Tree Soho bringing exotic Pan-Asian str...,https://maps.google.com/?cid=3467802162505449293,https://maps.google.com/?q=place_id:ChIJNyG8bd...,https://search.google.com/local/reviews?placei...,United Kingdom,1


In [43]:
# Hàm xử lý UK address
def robust_split_address_uk_v2(address, country="UK"):
    if pd.isna(address):
        return "", "", ""

    address = re.sub(
        rf'[,\s]*{re.escape(country)}[\s,\d]*$', '', address, flags=re.IGNORECASE
    ).strip()

    match = re.search(r'([A-Z]{1,2}\d[A-Z\d]?\s*\d[A-Z]{2})$', address, flags=re.IGNORECASE)
    if not match:
        return address, "", ""

    zip_code = match.group(1).strip()
    before_zip = address[:match.start()].strip().rstrip(", ")

    parts = [part.strip() for part in before_zip.split(",")]
    if len(parts) >= 2:
        city = parts[-1]
        street = ", ".join(parts[:-1])
    else:
        city = ""
        street = before_zip

    street_with_zip = f"{street}, {zip_code}"
    return street_with_zip, zip_code, city

# Áp dụng trực tiếp lên cột address
df_checkpoint[['street', 'zip', 'city']] = df_checkpoint['address'].apply(
    lambda x: pd.Series(robust_split_address_uk_v2(x))
)

# Xoá cột address
df_checkpoint.drop(columns=['address'], inplace=True)


In [44]:
df_checkpoint.head(5)

,post_title,latitude,longitude,phone,website,facebook,instagram,twitter,post_content,google_maps_link,google_profile,google_review_link,country,phone_status,street,zip,city
0,Viet Food,51.511621,-0.132167,+442074944555,http://www.vietnamfood.co.uk/,NaN,NaN,NaN,Close this moduleMain CourseFree Range Chicken...,https://maps.google.com/?cid=12549996454222453965,https://maps.google.com/?q=place_id:ChIJe6Owst...,https://search.google.com/local/reviews?placei...,United Kingdom,1,"34-36 Wardour St, W1D 6QT",W1D 6QT,London
1,Phat Phuc Noodle Bar,51.487544,-0.169204,+442073513843,http://www.phatphucnoodlebar.co.uk/,NaN,NaN,NaN,"Phat Phuc Noodle Bar located in The Courtyard,...",https://maps.google.com/?cid=18023201915501093142,https://maps.google.com/?q=place_id:ChIJwbvREG...,https://search.google.com/local/reviews?placei...,United Kingdom,1,"The Courtyard, 151 Sydney St, SW3 6NT",SW3 6NT,London
2,Union Viet Café,51.503872,-0.098374,+442033028828,http://www.unionviet.com/,NaN,NaN,NaN,Union Viet family run cafe & pho restaurant. ...,https://maps.google.com/?cid=5282475706193879885,https://maps.google.com/?q=place_id:ChIJuXu1JK...,https://search.google.com/local/reviews?placei...,United Kingdom,1,"120 Union St, SE1 0FR",SE1 0FR,London
3,Viet Baguette (Pho & Grill),51.488533,0.067871,+442088545588,NaN,NaN,NaN,NaN,Viet Baguette (Pho & Grill) located in 17 Angl...,https://maps.google.com/?cid=14466978752506452667,https://maps.google.com/?q=place_id:ChIJhcPehu...,https://search.google.com/local/reviews?placei...,United Kingdom,1,"17 Anglesea Rd, SE18 6EG",SE18 6EG,London
4,Banana Tree Soho,51.513099,-0.133865,+442074371351,https://bananatree.co.uk/restaurants/soho/?utm...,NaN,NaN,NaN,Banana Tree Soho bringing exotic Pan-Asian str...,https://maps.google.com/?cid=3467802162505449293,https://maps.google.com/?q=place_id:ChIJNyG8bd...,https://search.google.com/local/reviews?placei...,United Kingdom,1,"103 Wardour St, W1F 0UG",W1F 0UG,London


In [46]:
# Xuất file 
df_checkpoint.to_csv('uk_address_check.csv')

In [47]:
# Kiểm tra active của web 
import requests
from urllib.parse import urlparse
from concurrent.futures import ThreadPoolExecutor

# Chuẩn hóa URL
def normalize_url(url):
    if pd.isna(url) or not str(url).strip():
        return None
    url = url.strip()
    parsed = urlparse(url)
    if not parsed.scheme:
        return "http://" + url
    return url

# Kiểm tra hoạt động website, giữ original URL
def check_url(original_url):
    norm_url = normalize_url(original_url)
    if not norm_url:
        return (original_url, None, None, False)
    try:
        response = requests.get(norm_url, timeout=3, allow_redirects=True)
        final_url = response.url
        status = response.status_code
        is_active = 200 <= status < 400
        return (original_url, status, final_url, is_active)
    except:
        return (original_url, 0, None, False)

# Áp dụng đa luồng
df_checkpoint["normalized_url"] = df_checkpoint["website"].apply(normalize_url)
urls = df_checkpoint["normalized_url"].tolist()

with ThreadPoolExecutor(max_workers=30) as executor:
    results = list(executor.map(check_url, urls))

# Ghi kết quả vào DataFrame
df_checkpoint["web_status"] = [r[1] for r in results]
df_checkpoint["final_url"] = [r[2] for r in results]
df_checkpoint["is_active"] = [r[3] for r in results]


In [48]:
# Kiểm tra các link mạng xã hội bị lẫn trong website
# Xóa cột normalized_url
df_checkpoint.drop(columns=["normalized_url"], inplace=True)

# Xác định nền tảng mạng xã hội 
def classify_social_platform(url):
    if pd.isna(url):
        return None
    url = url.lower()
    if "facebook.com" in url:
        return "facebook"
    elif "instagram.com" in url:
        return "instagram"
    elif "twitter.com" in url or "x.com" in url:
        return "twitter"
    return None

df_checkpoint["social_platform"] = df_checkpoint["website"].apply(classify_social_platform)

# Chuyển các link sai về đúng cột
for platform in ["facebook", "instagram", "twitter"]:
    df_checkpoint[platform] = df_checkpoint.apply(
        lambda row: row["website"] if row["social_platform"] == platform and pd.isna(row[platform]) else row[platform],
        axis=1
    )

# Lưu kết quả vào file CSV
output_path = r"C:\Users\Nhung\Downloads\We_Love_Pho\Clean 25-5 - raw\uk_web_check.csv"

In [49]:
# Xoá khỏi giá trị cột website nếu là link MXH 
df_checkpoint.loc[df_checkpoint["social_platform"].notna(), "website"] = None
df_checkpoint.drop(columns=["social_platform"], inplace=True)

In [50]:
df_checkpoint.head(3)


,post_title,latitude,longitude,phone,website,facebook,instagram,twitter,post_content,google_maps_link,google_profile,google_review_link,country,phone_status,street,zip,city,web_status,final_url,is_active
0,Viet Food,51.511621,-0.132167,+442074944555,http://www.vietnamfood.co.uk/,NaN,NaN,NaN,Close this moduleMain CourseFree Range Chicken...,https://maps.google.com/?cid=12549996454222453965,https://maps.google.com/?q=place_id:ChIJe6Owst...,https://search.google.com/local/reviews?placei...,United Kingdom,1,"34-36 Wardour St, W1D 6QT",W1D 6QT,London,200.0,https://vietnamfood.co.uk/,True
1,Phat Phuc Noodle Bar,51.487544,-0.169204,+442073513843,http://www.phatphucnoodlebar.co.uk/,NaN,NaN,NaN,"Phat Phuc Noodle Bar located in The Courtyard,...",https://maps.google.com/?cid=18023201915501093142,https://maps.google.com/?q=place_id:ChIJwbvREG...,https://search.google.com/local/reviews?placei...,United Kingdom,1,"The Courtyard, 151 Sydney St, SW3 6NT",SW3 6NT,London,403.0,http://www.phatphucnoodlebar.co.uk/,False
2,Union Viet Café,51.503872,-0.098374,+442033028828,http://www.unionviet.com/,NaN,NaN,NaN,Union Viet family run cafe & pho restaurant. ...,https://maps.google.com/?cid=5282475706193879885,https://maps.google.com/?q=place_id:ChIJuXu1JK...,https://search.google.com/local/reviews?placei...,United Kingdom,1,"120 Union St, SE1 0FR",SE1 0FR,London,200.0,http://www.unionviet.com/,True


In [51]:
# tạo copy
df_checkpoint_copy = df_checkpoint.copy()

In [52]:
# 7. Lấy danh sách cột chuẩn từ file gd
gd_columns = df_gd.columns.tolist()

# 8. Thêm các cột còn thiếu và gán giá trị rỗng
for col in gd_columns:
    if col not in df_checkpoint_copy.columns:
        df_checkpoint_copy[col] = ""

# 9. Gán giá trị mặc định
df_checkpoint_copy['post_status'] = 'publish'
df_checkpoint_copy['post_category'] = ',249,'
df_checkpoint_copy['default_category'] = '249'
df_checkpoint_copy['featured'] = '0'

# 10. Sắp xếp lại đúng thứ tự cột
df_checkpoint_copy = df_checkpoint_copy[gd_columns]

# 12. Lưu file đã chuẩn hoá
df_checkpoint_copy.to_csv("uk_standardized.csv", index=False)

# 13. (Tuỳ chọn) Xem thử kết quả
print(df_checkpoint_copy[['street', 'zip', 'city', 'country']].head())

                                  street       zip    city         country
0              34-36 Wardour St, W1D 6QT   W1D 6QT  London  United Kingdom
1  The Courtyard, 151 Sydney St, SW3 6NT   SW3 6NT  London  United Kingdom
2                  120 Union St, SE1 0FR   SE1 0FR  London  United Kingdom
3               17 Anglesea Rd, SE18 6EG  SE18 6EG  London  United Kingdom
4                103 Wardour St, W1F 0UG   W1F 0UG  London  United Kingdom


In [53]:
df_checkpoint_copy.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1915 entries, 0 to 1914
Data columns (total 30 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   ID                1915 non-null   object 
 1   post_title        1915 non-null   object 
 2   post_content      1915 non-null   object 
 3   post_status       1915 non-null   object 
 4   post_author       1915 non-null   object 
 5   post_type         1915 non-null   object 
 6   post_date         1915 non-null   object 
 7   post_modified     1915 non-null   object 
 8   post_tags         1915 non-null   object 
 9   post_category     1915 non-null   object 
 10  default_category  1915 non-null   object 
 11  featured          1915 non-null   object 
 12  street            1915 non-null   object 
 13  street2           1915 non-null   object 
 14  city              1915 non-null   object 
 15  region            1915 non-null   object 
 16  country           1915 non-null   object 


In [54]:
# in ra 5 giá trị đầu cột lat và long
print(df_checkpoint_copy[['latitude', 'longitude']].head())

    latitude  longitude
0  51.511621  -0.132167
1  51.487544  -0.169204
2  51.503872  -0.098374
3  51.488533   0.067871
4  51.513099  -0.133865


In [55]:
duplicates = df_checkpoint_copy[df_checkpoint_copy.duplicated(keep=False)]

In [56]:
duplicate_count = duplicates.shape[0]
print(f"Số lượng bản ghi trùng lặp: {duplicate_count}")    


Số lượng bản ghi trùng lặp: 0


In [57]:
# Đếm số lượng giá trị không null cho từng dòng
df_checkpoint_copy['non_null_count'] = df_checkpoint_copy.notnull().sum(axis=1)

# Sắp xếp theo các cột và theo số lượng giá trị không null giảm dần
df_sorted = df_checkpoint_copy.sort_values(by=['post_title', 'latitude', 'longitude', 'street', 'non_null_count'], ascending=[True, True, True, True, False])

# Xóa các dòng trùng hoàn toàn, giữ lại dòng có nhiều thông tin nhất
df_deduplicated = df_sorted.drop_duplicates(keep='first').drop(columns=['non_null_count'])

# Lưu kết quả ra file mới
output_path = "uk_no_dup.csv"
df_deduplicated.to_csv(output_path, index=False)


In [58]:
df_deduplicated.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1915 entries, 1125 to 1237
Data columns (total 30 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   ID                1915 non-null   object 
 1   post_title        1915 non-null   object 
 2   post_content      1915 non-null   object 
 3   post_status       1915 non-null   object 
 4   post_author       1915 non-null   object 
 5   post_type         1915 non-null   object 
 6   post_date         1915 non-null   object 
 7   post_modified     1915 non-null   object 
 8   post_tags         1915 non-null   object 
 9   post_category     1915 non-null   object 
 10  default_category  1915 non-null   object 
 11  featured          1915 non-null   object 
 12  street            1915 non-null   object 
 13  street2           1915 non-null   object 
 14  city              1915 non-null   object 
 15  region            1915 non-null   object 
 16  country           1915 non-null   object 
 1

Top 200 từ khóa phổ biến và tần suất

In [65]:
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer
# Kết hợp nội dung từ 2 cột post_title và post_content
text_data = df_deduplicated[['post_title']].fillna('').agg(' '.join, axis=1)

# Khởi tạo CountVectorizer để trích xuất từ khóa
vectorizer = CountVectorizer(stop_words='english', max_features=500)
X = vectorizer.fit_transform(text_data)

# Lấy ra từ và tần suất
keywords = vectorizer.get_feature_names_out()
frequencies = np.asarray(X.sum(axis=0)).flatten()

# Tạo dataframe kết quả
keywords_df = pd.DataFrame({'keyword': keywords, 'frequency': frequencies}).sort_values(by='frequency', ascending=False)

In [66]:
# Lưu kết quả vào file CSV
output_keywords_path = "uk_keywords.csv"
keywords_df.to_csv(output_keywords_path, index=False)

In [67]:
# Kiểm tra độ chính xác 
positive_keywords = [
    'vietnam', 'viet', 'việt', 'pho', 'bún', 'nem', 'saigon', "phở", 'sài gòn', 'hà nội', 'hanoi', 'bánh mì',
    'halong', 'huế', 'bánh', 'goi cuon', 'bun cha', 'banh', 'thang', 'nam', 'sen', 'hoan kiem', 'wietnam', 'vietnamese',
    'sapa', 'tre', 'ha long', 'ha noi', 'sai gon', 'sajgon', 'hoang', 'ha-noi', 'com tam', 'hai', 'hoan', 'bami',
    'long', 'binh', 'banh mi', 'sao mai', 'song lam', 'ngoc', 'phuong dong', 'linh', 'vietnamská', 'vietnameské', 'quán', 'anh',
    'vietnamskou', 'vietnamské jídlo', 'vietnamská restaurace', 'ngon', 'hoi an', 'quan', 'vina', 'bếp', 'long', 'nón', 'hà', 
    'vietfood', 'gao', 'mì', 'mộc', 'mai', 'thanh', 'cà', 'tuan', 'lá', 'rong', 'vietnamskou', 'vietnamu', 'vietnamesisches', 'chả',
    'vietnamesische küche', 'vietnamesisch', 'vietnamesischen', "baguette", "mì", "hội", "thêm", "moc"
]
negative_keywords = [
    'chinese', 'thai', 'japan', 'korean', 'fusion', 'asia', 'china','india', 'ramen', 'pasta', 'pizza', 'burger',
    'sushi', 'tapas', 'mexican', 'indian', 'kebab', 'italian', 'curry', "tai wan", "singapore", "malaysia", "korea",
    "hong kong", 'resort', 'hotel', 'pub', 'cafe', 'coffee', 'steak', 'park', 'inn', 'post', 'market', 'hall', 'bbq',
    'library', 'sandwich', 'cantonese', 'peking', 'thajská', 'thajské', 'banyan', 'guty', 'shanghai', 'shi', 'pizzerie',
    'bubble', 'kyoto', "mongolian", "indochine", 'nail', 'spa', 'massage', 'laden', 'shop', 'store', 'beauty', 'thailändische',
    'asiatisch', 'asiatischen', 'café', "wagamama", "tuk", "coffee", "beauty", "giggling", "thailand", "beijing", "club", "bangkok",
    "halah", "pub", "theatre", "hungary", "mandarin", "yang", "castle", "casino", "kfc"
]

# Bước 3: Tạo regex pattern
pattern_positive = re.compile('|'.join(positive_keywords), re.IGNORECASE)
pattern_negative = re.compile('|'.join(negative_keywords), re.IGNORECASE)

# Bước 4: Hàm gán tag
def tag_positive(text):
    if pd.isna(text):
        return ''
    return 'Vietnamese restaurant' if pattern_positive.search(text) else ''

def tag_negative(text):
    if pd.isna(text):
        return ''
    return 'Others' if pattern_negative.search(text) else ''

# Bước 5: Gán PositiveTag và NegativeTag
df_deduplicated['Pos'] = df_deduplicated.apply(
    lambda row: tag_positive(row['post_title']) or tag_positive(row['post_content']),
    axis=1
)

df_deduplicated['Neg'] = df_deduplicated.apply(
    lambda row: tag_negative(row['post_title']) or tag_negative(row['post_content']),
    axis=1
)

# Bước 6: Logic gán ReCheck?
def final_recheck_tag(row):
    if row['Pos'] != '' and row['Neg'] == '':
        return 'N'
    elif row['Pos'] == '' and row['Neg'] != '':
        return 'N'
    elif row['Pos'] == '' and row['Neg'] == '':
        return 'Y'
    else:
        return 'Y'

df_deduplicated['ReCheck?'] = df_deduplicated.apply(final_recheck_tag, axis=1)

In [68]:
# Tạo label mẫu
# Tạo 1 cột mới tên là rỗng mới là Y trong df_checkpoint
def assign_label(row):
    pos = row['Pos'] == 'Vietnamese restaurant'
    neg = row['Neg'] == 'Others'
    recheck = row['ReCheck?']

    if pos and not neg and recheck == 'N':
        return 1
    elif neg and not pos and recheck == 'N':
        return 0
    elif pos and neg and recheck == 'Y':
        return 0
    else:
        return ''

df_deduplicated['Y'] = df_deduplicated.apply(assign_label, axis=1)

df = df_deduplicated.copy()

# Xuất file để check manual
columns_to_export = [
    'post_title', 'post_content', 'website', 'google_profile', "Y", 'city'
]
df = df[columns_to_export]
df.to_csv('uk_labeled.csv', index=False)


In [69]:
df_deduplicated.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1915 entries, 1125 to 1237
Data columns (total 34 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   ID                1915 non-null   object 
 1   post_title        1915 non-null   object 
 2   post_content      1915 non-null   object 
 3   post_status       1915 non-null   object 
 4   post_author       1915 non-null   object 
 5   post_type         1915 non-null   object 
 6   post_date         1915 non-null   object 
 7   post_modified     1915 non-null   object 
 8   post_tags         1915 non-null   object 
 9   post_category     1915 non-null   object 
 10  default_category  1915 non-null   object 
 11  featured          1915 non-null   object 
 12  street            1915 non-null   object 
 13  street2           1915 non-null   object 
 14  city              1915 non-null   object 
 15  region            1915 non-null   object 
 16  country           1915 non-null   object 
 1

In [70]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1915 entries, 1125 to 1237
Data columns (total 6 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   post_title      1915 non-null   object
 1   post_content    1915 non-null   object
 2   website         1414 non-null   object
 3   google_profile  1915 non-null   object
 4   Y               1915 non-null   object
 5   city            1915 non-null   object
dtypes: object(6)
memory usage: 104.7+ KB


In [73]:
# Đọc file labeled
df_labeled = pd.read_csv('uk_labeled.csv')
# Kiểm tra số nhãn của từng unique giá trị trong cột 'Y'
print(df_labeled['Y'].value_counts(dropna=False))

Y
0.0    1147
1.0     397
NaN     371
Name: count, dtype: int64


In [74]:
print(df_deduplicated['Y'].value_counts(dropna=False))

Y
0    1147
1     397
      371
Name: count, dtype: int64


In [75]:
# Kiểm tra lại nhanh số lượng giá trị thiếu (NaN) sau khi chuyển đổi
missing_summary = df_deduplicated.isna().sum()
missing_summary

ID                     0
post_title             0
post_content           0
post_status            0
post_author            0
post_type              0
post_date              0
post_modified          0
post_tags              0
post_category          0
default_category       0
featured               0
street                 0
street2                0
city                   0
region                 0
country                0
zip                    0
latitude               0
longitude              0
website              501
neighbourhood          0
facebook            1821
instagram           1886
twitter             1912
phone                119
email                  0
logo                   0
google_profile         0
post_images            0
Pos                    0
Neg                    0
ReCheck?               0
Y                      0
dtype: int64

In [126]:
df_final = df_deduplicated.copy()

In [ ]:
df_final[['phone']].head(10)


,phone
1125,+441223505555
748,+441753866488
195,+447971088418
1514,NaN
194,+447806770926
1063,+441295279140
845,+442072473344
494,+442081766951
1849,NaN
280,+447448150081


In [128]:
df_final['phone'] = df_final['phone'].apply(lambda x: str(int(x)) if pd.notna(x) else "")
df_final[['phone']].head(10)

,phone
1125,441223505555
748,441753866488
195,447971088418
1514,
194,447806770926
1063,441295279140
845,442072473344
494,442081766951
1849,
280,447448150081


In [129]:
df_final.head(5)

,ID,post_title,post_content,post_status,post_author,post_type,post_date,post_modified,post_tags,post_category,...,twitter,phone,email,logo,google_profile,post_images,Pos,Neg,ReCheck?,Y
1125,,1+1 Rougamo,"1+1 Rougamo located in 84 Regent St, Cambridge...",publish,,,,,,",249,",...,NaN,441223505555,,,https://maps.google.com/?q=place_id:ChIJpYWmfK...,,,,Y,
748,,1423 China Kitchen,LoginorRegisterHomeMenuTake-away MenuDine-in M...,publish,,,,,,",249,",...,NaN,441753866488,,,https://maps.google.com/?q=place_id:ChIJ30hsre...,,,Others,N,0
195,,168 Oriental Supermarket,168 Oriental Supermarket located in 1 Summerla...,publish,,,,,,",249,",...,NaN,447971088418,,,https://maps.google.com/?q=place_id:ChIJJZqZoR...,,,Others,N,0
1514,,2C9 Asian Street Food,"2C9 Asian Street Food located in Unit B2, Indo...",publish,,,,,,",249,",...,NaN,,,,https://maps.google.com/?q=place_id:ChIJs9fmt5...,,Vietnamese restaurant,Others,Y,0
194,,3 East Street,"3 East Street located in 3 East St, Okehampton...",publish,,,,,,",249,",...,NaN,447806770926,,,https://maps.google.com/?q=place_id:ChIJ1UQBU3...,,Vietnamese restaurant,,N,1


In [103]:
# Gán giá trị mặc định nếu thiếu
df_final['post_type'] = df_final['post_type'].fillna('gd_place')
# Gán ngày mặc định nếu thiếu, đảm bảo đúng định dạng chuỗi
default_date = '2025-06-01 00:00:00'
df_final['post_date'] = df_final['post_date'].fillna(default_date)
df_final['post_modified'] = df_final['post_modified'].fillna(default_date)
df_final['post_author'] = df_final['post_author'].fillna('admin')
df_final["post_content"] = ""
df_final = df_final.replace(r'^\s*$', pd.NA, regex=True)
df_final.set_index('ID', inplace=True)
# Đổi tên cột id thành ID
df_final.head(5)
df_final.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1915 entries, <NA> to <NA>
Data columns (total 33 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   post_title        1915 non-null   object 
 1   post_content      0 non-null      object 
 2   post_status       1915 non-null   object 
 3   post_author       0 non-null      object 
 4   post_type         0 non-null      object 
 5   post_date         0 non-null      object 
 6   post_modified     0 non-null      object 
 7   post_tags         0 non-null      object 
 8   post_category     1915 non-null   object 
 9   default_category  1915 non-null   object 
 10  featured          1915 non-null   object 
 11  street            1915 non-null   object 
 12  street2           0 non-null      object 
 13  city              1906 non-null   object 
 14  region            0 non-null      object 
 15  country           1915 non-null   object 
 16  zip               1914 non-null   object 
 1

In [130]:
df_final[['phone']].head(10)

,phone
1125,441223505555
748,441753866488
195,447971088418
1514,
194,447806770926
1063,441295279140
845,442072473344
494,442081766951
1849,
280,447448150081


In [135]:
df_final.head(5)
df_final.to_csv('uk_final_final.csv', index=False)

In [108]:
%pip install openpyxl --quiet

# Xuất file excel
output_excel_path = 'uk_final_final.csv'
df_final.to_excel(output_excel_path, index=False)

Note: you may need to restart the kernel to use updated packages.


ValueError: No engine for filetype: 'csv'

In [136]:
# đọc file labeled_new
#df_labeled_new = pd.read_csv('ger_labeled_new.csv')
df_standardized= pd.read_csv('uk_final_final.csv')
# Kiểm tra số nhãn của từng unique giá trị trong cột 'Y'
print(df_standardized['Y'].value_counts(dropna=False))

Y
0.0    1147
1.0     397
NaN     371
Name: count, dtype: int64


In [138]:
df_standardized['phone'] = df_standardized['phone'].apply(lambda x: str(int(x)) if pd.notna(x) else "")
df_standardized[['phone']].head(10)

,phone
0,441223505555
1,441753866488
2,447971088418
3,
4,447806770926
5,441295279140
6,442072473344
7,442081766951
8,
9,447448150081


In [139]:
df_standardized.head(5)

,ID,post_title,post_content,post_status,post_author,post_type,post_date,post_modified,post_tags,post_category,...,twitter,phone,email,logo,google_profile,post_images,Pos,Neg,ReCheck?,Y
0,NaN,1+1 Rougamo,"1+1 Rougamo located in 84 Regent St, Cambridge...",publish,NaN,NaN,NaN,NaN,NaN,",249,",...,NaN,441223505555,NaN,NaN,https://maps.google.com/?q=place_id:ChIJpYWmfK...,NaN,NaN,NaN,Y,NaN
1,NaN,1423 China Kitchen,LoginorRegisterHomeMenuTake-away MenuDine-in M...,publish,NaN,NaN,NaN,NaN,NaN,",249,",...,NaN,441753866488,NaN,NaN,https://maps.google.com/?q=place_id:ChIJ30hsre...,NaN,NaN,Others,N,0.0
2,NaN,168 Oriental Supermarket,168 Oriental Supermarket located in 1 Summerla...,publish,NaN,NaN,NaN,NaN,NaN,",249,",...,NaN,447971088418,NaN,NaN,https://maps.google.com/?q=place_id:ChIJJZqZoR...,NaN,NaN,Others,N,0.0
3,NaN,2C9 Asian Street Food,"2C9 Asian Street Food located in Unit B2, Indo...",publish,NaN,NaN,NaN,NaN,NaN,",249,",...,NaN,,NaN,NaN,https://maps.google.com/?q=place_id:ChIJs9fmt5...,NaN,Vietnamese restaurant,Others,Y,0.0
4,NaN,3 East Street,"3 East Street located in 3 East St, Okehampton...",publish,NaN,NaN,NaN,NaN,NaN,",249,",...,NaN,447806770926,NaN,NaN,https://maps.google.com/?q=place_id:ChIJ1UQBU3...,NaN,Vietnamese restaurant,NaN,N,1.0


In [140]:
# Bổ sung giá trị cột Y từ df_labeled_new vào df_standardized dựa trên index (không có cột ID)
df_labeled_new = pd.read_csv(r"C:\Users\Nhung\Downloads\We_Love_Pho\uk_final.csv")
df_standardized['Y'] = df_labeled_new['Y'].values
# Lưu kết quả vào file CSV
output_path = 'uk_standardized_with_labels.csv'
# đọc poland_standardized_with_labels.csv
df_standardized.to_csv(output_path, index=False)
df_standardized_with_labels = pd.read_csv(output_path)  
# Kiểm tra số nhãn của từng unique giá trị trong cột 'Y'
print(df_standardized_with_labels['Y'].value_counts(dropna=False))
# Lưu lại file đã chuẩn hoá
df_standardized_with_labels.to_csv('uk_standardized_final.csv', index=False)
# in head 5 dòng
df_standardized_with_labels.head(5)

Y
0.0    1066
NaN     468
1.0     381
Name: count, dtype: int64


,ID,post_title,post_content,post_status,post_author,post_type,post_date,post_modified,post_tags,post_category,...,twitter,phone,email,logo,google_profile,post_images,Pos,Neg,ReCheck?,Y
0,NaN,1+1 Rougamo,"1+1 Rougamo located in 84 Regent St, Cambridge...",publish,NaN,NaN,NaN,NaN,NaN,",249,",...,NaN,4.412235e+11,NaN,NaN,https://maps.google.com/?q=place_id:ChIJpYWmfK...,NaN,NaN,NaN,Y,NaN
1,NaN,1423 China Kitchen,LoginorRegisterHomeMenuTake-away MenuDine-in M...,publish,NaN,NaN,NaN,NaN,NaN,",249,",...,NaN,4.417539e+11,NaN,NaN,https://maps.google.com/?q=place_id:ChIJ30hsre...,NaN,NaN,Others,N,0.0
2,NaN,168 Oriental Supermarket,168 Oriental Supermarket located in 1 Summerla...,publish,NaN,NaN,NaN,NaN,NaN,",249,",...,NaN,4.479711e+11,NaN,NaN,https://maps.google.com/?q=place_id:ChIJJZqZoR...,NaN,NaN,Others,N,0.0
3,NaN,2C9 Asian Street Food,"2C9 Asian Street Food located in Unit B2, Indo...",publish,NaN,NaN,NaN,NaN,NaN,",249,",...,NaN,NaN,NaN,NaN,https://maps.google.com/?q=place_id:ChIJs9fmt5...,NaN,Vietnamese restaurant,Others,Y,0.0
4,NaN,3 East Street,"3 East Street located in 3 East St, Okehampton...",publish,NaN,NaN,NaN,NaN,NaN,",249,",...,NaN,4.478068e+11,NaN,NaN,https://maps.google.com/?q=place_id:ChIJ1UQBU3...,NaN,Vietnamese restaurant,NaN,N,NaN


In [142]:
df_standardized_with_labels['phone'] = df_standardized_with_labels['phone'].apply(lambda x: str(int(x)) if pd.notna(x) else "")
df_standardized_with_labels[['phone']].head(10)

,phone
0,441223505555
1,441753866488
2,447971088418
3,
4,447806770926
5,441295279140
6,442072473344
7,442081766951
8,
9,447448150081


In [143]:
df_standardized = df_standardized_with_labels.copy()

In [112]:
# Gán giá trị mặc định nếu thiếu
df_standardized['post_type'] = df_standardized['post_type'].fillna('gd_place')
# Gán ngày mặc định nếu thiếu, đảm bảo đúng định dạng chuỗi
default_date = '2025-05-31 00:00:00'
df_standardized['post_date'] = df_standardized['post_date'].fillna(default_date)
df_standardized['post_modified'] = df_standardized['post_modified'].fillna(default_date)
df_standardized['post_author'] = df_standardized['post_author'].fillna('admin')
df_standardized["post_content"] = ""
df_standardized = df_standardized.replace(r'^\s*$', pd.NA, regex=True)
# trích xuất file cuối chỉ có các dòng mà giá trị cột Y là 1 và xóa cột Y sau đó 
df_standardized = df_standardized[df_standardized['Y'] == 1]
df_standardized.drop(columns=['Y'], inplace=True)
df_standardized.head(5)

,post_title,post_content,post_status,post_author,post_type,post_date,post_modified,post_tags,post_category,default_category,...,instagram,twitter,phone,email,logo,google_profile,post_images,Pos,Neg,ReCheck?
6,3 Mien,<NA>,publish,admin,gd_place,2025-05-31 00:00:00,2025-05-31 00:00:00,NaN,",249,",249,...,NaN,NaN,4.420720e+11,NaN,NaN,https://maps.google.com/?q=place_id:ChIJud1qFx...,NaN,NaN,NaN,Y
8,4 Seasons Tree,<NA>,publish,admin,gd_place,2025-05-31 00:00:00,2025-05-31 00:00:00,NaN,",249,",249,...,NaN,NaN,NaN,NaN,NaN,https://maps.google.com/?q=place_id:ChIJwxGx_I...,NaN,Vietnamese restaurant,NaN,N
29,Amie's Kitchen,<NA>,publish,admin,gd_place,2025-05-31 00:00:00,2025-05-31 00:00:00,NaN,",249,",249,...,NaN,NaN,4.478780e+11,NaN,NaN,https://maps.google.com/?q=place_id:ChIJK_BLGj...,NaN,Vietnamese restaurant,NaN,N
30,Amthuc Viet,<NA>,publish,admin,gd_place,2025-05-31 00:00:00,2025-05-31 00:00:00,NaN,",249,",249,...,NaN,NaN,4.479000e+11,NaN,NaN,https://maps.google.com/?q=place_id:ChIJNxj_s3...,NaN,Vietnamese restaurant,NaN,N
31,An Nam Restaurant,<NA>,publish,admin,gd_place,2025-05-31 00:00:00,2025-05-31 00:00:00,NaN,",249,",249,...,NaN,NaN,4.420810e+11,NaN,NaN,https://maps.google.com/?q=place_id:ChIJ8yXry4...,NaN,Vietnamese restaurant,Others,Y


In [113]:
# So sánh định dạng từng cột của df_true với df_gd
def compare_column_formats(df1, df2):
    comparison = {}
    for col in df1.columns:
        if col in df2.columns:
            comparison[col] = {
                'df1_dtype': df1[col].dtype,
                'df2_dtype': df2[col].dtype,
                'df1_unique_count': df1[col].nunique(),
                'df2_unique_count': df2[col].nunique()
            }
        else:
            comparison[col] = {
                'df1_dtype': df1[col].dtype,
                'df2_dtype': None,
                'df1_unique_count': df1[col].nunique(),
                'df2_unique_count': None
            }
    return comparison
# So sánh định dạng cột của df_true với df_gd
comparison_result = compare_column_formats(df_standardized, df_gd)
# In kết quả so sánh
for col, info in comparison_result.items():
    print(f"Cột: {col}")
    print(f"  - df_true dtype: {info['df1_dtype']}, unique count: {info['df1_unique_count']}")
    print(f"  - df_gd dtype: {info['df2_dtype']}, unique count: {info['df2_unique_count']}")
    print()
# Ép kiểu các cột trong df_true để phù hợp với df_gd
def convert_column_types(df, reference_df):
    for col in reference_df.columns:
        if col in df.columns:
            ref_dtype = reference_df[col].dtype
            if ref_dtype == 'object':
                df[col] = df[col].astype(str)
            elif ref_dtype == 'int64':
                df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0).astype(int)
            elif ref_dtype == 'float64':
                df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0.0).astype(float)
            elif ref_dtype == 'datetime64[ns]':
                df[col] = pd.to_datetime(df[col], errors='coerce')
    return df   
# Chuyển đổi kiểu dữ liệu của df_true để phù hợp với df_gd
df_standardized = convert_column_types(df_standardized, df_gd)

Cột: post_title
  - df_true dtype: object, unique count: 318
  - df_gd dtype: object, unique count: 100

Cột: post_content
  - df_true dtype: object, unique count: 0
  - df_gd dtype: object, unique count: 19

Cột: post_status
  - df_true dtype: object, unique count: 1
  - df_gd dtype: object, unique count: 1

Cột: post_author
  - df_true dtype: object, unique count: 1
  - df_gd dtype: int64, unique count: 19

Cột: post_type
  - df_true dtype: object, unique count: 1
  - df_gd dtype: object, unique count: 1

Cột: post_date
  - df_true dtype: object, unique count: 1
  - df_gd dtype: object, unique count: 100

Cột: post_modified
  - df_true dtype: object, unique count: 1
  - df_gd dtype: object, unique count: 100

Cột: post_tags
  - df_true dtype: float64, unique count: 0
  - df_gd dtype: object, unique count: 1

Cột: post_category
  - df_true dtype: object, unique count: 1
  - df_gd dtype: object, unique count: 1

Cột: default_category
  - df_true dtype: int64, unique count: 1
  - df_gd 

In [144]:
# trích xuất file cuối chỉ có các dòng mà giá trị cột Y là 1 và xóa cột Y sau đó 
df_standardized = df_standardized[df_standardized['Y'] == 1]
df_standardized.drop(columns=['Y'], inplace=True)
df_standardized.head(5)

,ID,post_title,post_content,post_status,post_author,post_type,post_date,post_modified,post_tags,post_category,...,instagram,twitter,phone,email,logo,google_profile,post_images,Pos,Neg,ReCheck?
6,NaN,3 Mien,"3 Mien located in 64 Middlesex St, London E1 7...",publish,NaN,NaN,NaN,NaN,NaN,",249,",...,NaN,NaN,442072473344,NaN,NaN,https://maps.google.com/?q=place_id:ChIJud1qFx...,NaN,NaN,NaN,Y
8,NaN,4 Seasons Tree,"4 Seasons Tree located in 52 North Rd, Durham ...",publish,NaN,NaN,NaN,NaN,NaN,",249,",...,NaN,NaN,,NaN,NaN,https://maps.google.com/?q=place_id:ChIJwxGx_I...,NaN,Vietnamese restaurant,NaN,N
29,NaN,Amie's Kitchen,Amies Kitchen Vietnamese restaurant dine in an...,publish,NaN,NaN,NaN,NaN,NaN,",249,",...,NaN,NaN,447877598003,NaN,NaN,https://maps.google.com/?q=place_id:ChIJK_BLGj...,NaN,Vietnamese restaurant,NaN,N
30,NaN,Amthuc Viet,Home of Amthuc Viet. A Vietnamese catering & t...,publish,NaN,NaN,NaN,NaN,NaN,",249,",...,NaN,NaN,447900307025,NaN,NaN,https://maps.google.com/?q=place_id:ChIJNxj_s3...,NaN,Vietnamese restaurant,NaN,N
31,NaN,An Nam Restaurant,HomeAbout UsMenuGalleryReservationContact UsHo...,publish,NaN,NaN,NaN,NaN,NaN,",249,",...,NaN,NaN,442081434225,NaN,NaN,https://maps.google.com/?q=place_id:ChIJ8yXry4...,NaN,Vietnamese restaurant,Others,Y


In [145]:
df_standardized[['phone']].head(10)

,phone
6,442072473344
8,
29,447877598003
30,447900307025
31,442081434225
68,447950218527
70,447539184534
92,441206866775
93,442088913229
94,


In [119]:
def check_duplicates_multiple_columns(df1, df2, columns):
    merged = df1.merge(df2[columns].drop_duplicates(), on=columns, how='inner')
    return merged

# Gọi với 2 cột
duplicates = check_duplicates_multiple_columns(df_standardized, df_gd, [ 'zip'])

print(f"Số lượng bản ghi trùng lặp theo 'post_title' và 'country': {duplicates.shape[0]}")
print(duplicates[['post_title', 'zip']].head(5))


Số lượng bản ghi trùng lặp theo 'post_title' và 'country': 17
                            post_title       zip
0                              Cay Khe  OX14 3JF
1          Nón - Vietnamese restaurant    N7 8JP
2  Nón express - Vietnamese restaurant  EC1V 3RA
3                                  Pho   CB2 3QB
4                            Pho & Bun   W1D 6ND


In [49]:
# Xóa các bản ghi trùng lặp trong df_standardized_with_labels
df_standardized = df_standardized[~df_standardized['post_title'].isin(duplicates['post_title'])]
# In ra số lượng bản ghi sau khi xóa trùng lặp
print(f"Số lượng bản ghi sau khi xóa trùng lặp: {df_standardized.shape[0]}")

Số lượng bản ghi sau khi xóa trùng lặp: 1363


In [146]:
df_standardized.head(5)
#Xóa cột neg, pos, recheck
df_standardized.drop(columns=['Neg', 'Pos', 'ReCheck?'], inplace=True)
# In ra thông tin của các cột
df_standardized.info()
# Lưu lại file đã chuẩn hoá
df_standardized.to_csv('uk_upload.csv', index=False)

<class 'pandas.core.frame.DataFrame'>
Index: 381 entries, 6 to 1911
Data columns (total 30 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   ID                0 non-null      float64
 1   post_title        381 non-null    object 
 2   post_content      381 non-null    object 
 3   post_status       381 non-null    object 
 4   post_author       0 non-null      float64
 5   post_type         0 non-null      float64
 6   post_date         0 non-null      float64
 7   post_modified     0 non-null      float64
 8   post_tags         0 non-null      float64
 9   post_category     381 non-null    object 
 10  default_category  381 non-null    int64  
 11  featured          381 non-null    int64  
 12  street            381 non-null    object 
 13  street2           0 non-null      float64
 14  city              379 non-null    object 
 15  region            0 non-null      float64
 16  country           381 non-null    object 
 17  z

In [153]:
import pandas as pd

# 1. Load dữ liệu
df_sample = pd.read_csv(r"C:\Users\Nhung\Downloads\We_Love_Pho\sample structure.csv")

# 2. Thêm cột ID từ index (bắt đầu từ 1)
df_standardized['ID'] = df_standardized.index + 1
df_standardized = df_standardized[['ID'] + [col for col in df_standardized.columns if col != 'ID']]

# 3. Đảm bảo các cột đúng thứ tự như file mẫu
df_standardized = df_standardized[df_sample.columns]
df_standardized = df_standardized.replace(r'^\s*$', pd.NA, regex=True)
df_standardized["post_type"] = "gd_place"
# Gán ngày mặc định nếu thiếu, đảm bảo đúng định dạng chuỗi
df_standardized['post_date'] = '2025-06-03 00:00:00'
df_standardized['post_modified'] = '2025-06-03 00:00:00'
df_standardized['post_author'] = 'admin'
df_standardized["post_content"] = ""
df_standardized = df_standardized.replace(r'^\s*$', pd.NA, regex=True)
# 4. Chuyển kiểu dữ liệu theo sample
for col in df_sample.columns:
    ref_dtype = df_sample[col].dtype
    if ref_dtype == 'object':
        df_standardized[col] = df_standardized[col].astype(str)
    elif 'int' in str(ref_dtype):
        df_standardized[col] = pd.to_numeric(df_standardized[col], errors='coerce').fillna(0).astype(int)
    elif 'float' in str(ref_dtype):
        df_standardized[col] = pd.to_numeric(df_standardized[col], errors='coerce')
    elif 'datetime' in str(ref_dtype):
        df_standardized[col] = pd.to_datetime(df_standardized[col], errors='coerce')

# 5. Làm sạch các chuỗi rỗng hoặc chứa 'nan', 'none'
df_standardized = df_standardized.replace(r'^\s*$', pd.NA, regex=True)
df_standardized = df_standardized.applymap(lambda x: pd.NA if isinstance(x, str) and x.strip().lower() in ['nan', 'none'] else x)

df_standardized['region'] = df_standardized['region'].apply(
    lambda x: "-" if pd.isna(x) or str(x).strip().lower() in ['0.0', 'nan', 'none', 'n/a'] else x
)
# Chuẩn hóa zip code
df_standardized['zip'] = df_standardized['zip'].astype(str).str.replace(r'\.0$', '', regex=True)
df_standardized['zip'] = df_standardized['zip'].apply(lambda x: x.zfill(5) if x.isdigit() else x)
# Làm sạch street2 để không có 0.0 hoặc NaN
df_standardized['street2'] = df_standardized['street2'].apply(
    lambda x: pd.NA if pd.isna(x) or str(x).strip().lower() in ['0.0', 'nan', 'none'] else x
)

# 6. Chuẩn hóa số điện thoại
df_standardized['phone'] = df_standardized['phone'].astype(str).str.replace(r'\.0$', '', regex=True)
df_standardized['phone'] = df_standardized['phone'].apply(lambda x: '+' + x if isinstance(x, str) and x and not x.startswith('+') else x)

df_standardized.drop(columns=['ID'], inplace=True)
# 8. Xuất ra file CSV
df_standardized.to_csv("ik_final_upload_ready.csv", index=False)


C:\Users\Nhung\AppData\Local\Temp\ipykernel_35168\2317629895.py:34: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df_standardized = df_standardized.applymap(lambda x: pd.NA if isinstance(x, str) and x.strip().lower() in ['nan', 'none'] else x)


In [151]:
df_standardized.head(5)

,post_title,post_content,post_status,post_author,post_type,post_date,post_modified,post_tags,post_category,default_category,...,website,neighbourhood,facebook,instagram,twitter,phone,email,logo,google_profile,post_images
6,3 Mien,<NA>,publish,0,gd_place,2025-06-03 00:00:00,2025-06-03 00:00:00,<NA>,",249,",249,...,<NA>,NaN,<NA>,<NA>,<NA>,+442072473344,<NA>,<NA>,https://maps.google.com/?q=place_id:ChIJud1qFx...,NaN
8,4 Seasons Tree,<NA>,publish,0,gd_place,2025-06-03 00:00:00,2025-06-03 00:00:00,<NA>,",249,",249,...,<NA>,NaN,<NA>,<NA>,<NA>,+<NA>,<NA>,<NA>,https://maps.google.com/?q=place_id:ChIJwxGx_I...,NaN
29,Amie's Kitchen,<NA>,publish,0,gd_place,2025-06-03 00:00:00,2025-06-03 00:00:00,<NA>,",249,",249,...,https://www.amieskitchen.net/,NaN,<NA>,<NA>,<NA>,+447877598003,<NA>,<NA>,https://maps.google.com/?q=place_id:ChIJK_BLGj...,NaN
30,Amthuc Viet,<NA>,publish,0,gd_place,2025-06-03 00:00:00,2025-06-03 00:00:00,<NA>,",249,",249,...,http://amthuc-viet.co.uk/,NaN,<NA>,<NA>,<NA>,+447900307025,<NA>,<NA>,https://maps.google.com/?q=place_id:ChIJNxj_s3...,NaN
31,An Nam Restaurant,<NA>,publish,0,gd_place,2025-06-03 00:00:00,2025-06-03 00:00:00,<NA>,",249,",249,...,https://annamrestaurant.co.uk/,NaN,<NA>,<NA>,<NA>,+442081434225,<NA>,<NA>,https://maps.google.com/?q=place_id:ChIJ8yXry4...,NaN


In [160]:
import pandas as pd

# 1. Load dữ liệu
df_slovakia = pd.read_csv(r"C:\Users\Nhung\Downloads\We_Love_Pho\uk_upload.csv")
df_sample = pd.read_csv(r"C:\Users\Nhung\Downloads\We_Love_Pho\sample structure.csv")

# 2. Thêm cột ID từ index (bắt đầu từ 1)
df_slovakia['ID'] = df_slovakia.index + 1
df_slovakia = df_slovakia[['ID'] + [col for col in df_slovakia.columns if col != 'ID']]

# 3. Đảm bảo các cột đúng thứ tự như file mẫu
df_slovakia = df_slovakia[df_sample.columns]

df_slovakia["post_type"] = "gd_place"
# Gán ngày mặc định nếu thiếu, đảm bảo đúng định dạng chuỗi
df_slovakia['post_date'] = '2025-06-03 00:00:00'
df_slovakia['post_modified'] = '2025-06-03 00:00:00'
df_slovakia['post_author'] = 'admin'
df_slovakia["post_content"] = ''

# 4. Chuyển kiểu dữ liệu theo sample
for col in df_sample.columns:
    ref_dtype = df_sample[col].dtype
    if ref_dtype == 'object':
        df_slovakia[col] = df_slovakia[col].astype(str)
    elif 'int' in str(ref_dtype):
        df_slovakia[col] = pd.to_numeric(df_slovakia[col], errors='coerce').fillna(0).astype(int)
    elif 'float' in str(ref_dtype):
        df_slovakia[col] = pd.to_numeric(df_slovakia[col], errors='coerce')
    elif 'datetime' in str(ref_dtype):
        df_slovakia[col] = pd.to_datetime(df_slovakia[col], errors='coerce')

# 5. Làm sạch các chuỗi rỗng hoặc chứa 'nan', 'none'
df_slovakia = df_slovakia.replace(r'^\s*$', pd.NA, regex=True)
df_slovakia = df_slovakia.applymap(lambda x: pd.NA if isinstance(x, str) and x.strip().lower() in ['nan', 'none'] else x)

df_slovakia['region'] = df_slovakia['region'].apply(
    lambda x: "-" if pd.isna(x) or str(x).strip().lower() in ['0.0', 'nan', 'none', 'n/a'] else x
)
# Chuẩn hóa zip code
df_slovakia['zip'] = df_slovakia['zip'].astype(str).str.replace(r'\.0$', '', regex=True)
df_slovakia['zip'] = df_slovakia['zip'].apply(lambda x: x.zfill(5) if x.isdigit() else x)
# Làm sạch street2 để không có 0.0 hoặc NaN
df_slovakia['street2'] = df_slovakia['street2'].apply(
    lambda x: pd.NA if pd.isna(x) or str(x).strip().lower() in ['0.0', 'nan', 'none'] else x
)

# 6. Chuẩn hóa số điện thoại
df_slovakia['phone'] = df_slovakia['phone'].astype(str).str.replace(r'\.0$', '', regex=True)
df_slovakia['phone'] = df_slovakia['phone'].apply(lambda x: '+' + x if isinstance(x, str) and x and not x.startswith('+') else x)

df_slovakia.drop(columns=['ID'], inplace=True)
# 8. Xuất ra file CSV
df_slovakia.to_csv("ik_final_upload_ready.csv", index=False)


C:\Users\Nhung\AppData\Local\Temp\ipykernel_35168\2430832448.py:35: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df_slovakia = df_slovakia.applymap(lambda x: pd.NA if isinstance(x, str) and x.strip().lower() in ['nan', 'none'] else x)


In [161]:

# 1. Load dữ liệu
df_sample = pd.read_csv(r"C:\Users\Nhung\Downloads\We_Love_Pho\sample structure.csv")
# Kiểm tra trùng post_title giữa df_standardized_with_labels và df_gd
def check_duplicates(df1, df2, column):
    duplicates = df1[df1[column].isin(df2[column])]
    return duplicates
# Kiểm tra trùng post_title
duplicates = check_duplicates(df_slovakia, df_sample, 'zip')
# In ra số lượng bản ghi trùng lặp
print(f"Số lượng bản ghi trùng lặp trong cột 'post_title': {duplicates.shape[0]}")
# In ra 5 bản ghi trùng lặp
print(duplicates[['post_title']].head(5))

Số lượng bản ghi trùng lặp trong cột 'post_title': 17
                              post_title
49                               Cay Khe
152          Nón - Vietnamese restaurant
153  Nón express - Vietnamese restaurant
166                                  Pho
216                            Pho & Bun


In [162]:
# Xóa các bản ghi trùng lặp trong df_standardized_with_labels
df_slovakia = df_slovakia[~df_slovakia['zip'].isin(duplicates['zip'])]
# In ra số lượng bản ghi sau khi xóa trùng lặp
print(f"Số lượng bản ghi sau khi xóa trùng lặp: {df_slovakia.shape[0]}")

Số lượng bản ghi sau khi xóa trùng lặp: 364


In [163]:
df_slovakia.to_csv("ik_final_upload_ready.csv", index=False)